# Python Coding Round — the 20 Patterns That Cover Most Screens

ML coding screens don't draw from a random pool — the same ~20 patterns recur endlessly. Each pattern below is one representative problem with an executed solution. Every solution is followed by asserts that **pass when you run this notebook**, so what you rehearse here is verified, not vibes.

**How to rehearse:** read the problem, write your own solution on paper or a blank editor *before* unfolding the answer, then compare — especially the complexity discussion, which is where the interviewer's attention actually is.

## Pattern 1 — Two Pointers: Pair With Target Sum

*Given a sorted array, return the indices of two numbers adding to a target. O(n) or better.*

Two pointers converge from both ends. Sortedness is the enabler: if the sum is too small, only moving the left pointer up can help; too large, only the right down. This replaces the O(n²) double loop.

In [1]:
def pair_with_target(arr, target):
    left, right = 0, len(arr) - 1
    while left < right:
        s = arr[left] + arr[right]
        if s == target:
            return (left, right)
        if s < target:
            left += 1
        else:
            right -= 1
    return None

assert pair_with_target([1, 2, 3, 4, 6], 6) == (1, 3)
assert pair_with_target([2, 5, 9, 11], 11) == (0, 2)
assert pair_with_target([1, 2, 3], 100) is None
print("pattern 1 verified — O(n) time, O(1) space")

pattern 1 verified — O(n) time, O(1) space


## Pattern 2 — Sliding Window: Longest Substring Without Repeats

*Find the length of the longest substring with no repeated character.*

The window `[start, i]` holds the current valid substring. When a repeat appears, jump `start` past the previous occurrence. Each character enters and leaves the window at most once → O(n), which is the sentence to say out loud.

In [2]:
def longest_unique(s):
    seen, start, best = {}, 0, 0
    for i, ch in enumerate(s):
        if ch in seen and seen[ch] >= start:
            start = seen[ch] + 1
        seen[ch] = i
        best = max(best, i - start + 1)
    return best

assert longest_unique("abcabcbb") == 3
assert longest_unique("bbbbb") == 1
assert longest_unique("pwwkew") == 3
assert longest_unique("") == 0
print("pattern 2 verified — O(n) time, O(min(charset, n)) space")

pattern 2 verified — O(n) time, O(min(charset, n)) space


## Pattern 3 — Prefix Sums: Range Sum in O(1)

*Answer many "sum of arr[i..j]" queries fast.* Precompute cumulative sums once; each query is a subtraction. This is also the skeleton of moving-average and elevation-rain problems.

In [3]:
class PrefixSum:
    def __init__(self, arr):
        self.pre = [0]
        for x in arr:
            self.pre.append(self.pre[-1] + x)
    def range_sum(self, i, j):        # inclusive
        return self.pre[j + 1] - self.pre[i]

ps = PrefixSum([2, 4, 1, 7, 3, 9])
assert ps.range_sum(1, 3) == 12
assert ps.range_sum(0, 5) == 26
assert ps.range_sum(2, 2) == 1
print("pattern 3 verified — O(n) build, O(1) per query")

pattern 3 verified — O(n) build, O(1) per query


## Pattern 4 — Hash Map Counting: Top-K Frequent

*Return the k most frequent elements.* Counter does the counting; a heap of size k keeps the top ones without a full sort. Mention `nlargest` is O(n log k), better than sorting all keys O(n log n) when k is small.

In [4]:
from collections import Counter
import heapq

def top_k_frequent(items, k):
    counts = Counter(items)
    return [x for x, _ in heapq.nlargest(k, counts.items(), key=lambda kv: kv[1])]

assert top_k_frequent([1,1,1,2,2,3], 2) == [1, 2]
assert top_k_frequent(["a","b","b","c","c","c"], 1) == ["c"]
print("pattern 4 verified — O(n log k)")

pattern 4 verified — O(n log k)


## Pattern 5 — Binary Search on the Answer: First Bad Version

The subtle half of binary search: the array isn't given — the *monotone predicate* is. "Bad" is False...False,True...True; find the first True. This pattern solves "minimum capacity to ship in D days", "smallest divisor ≥ threshold", etc.

In [5]:
def first_bad_version(n, is_bad):
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if is_bad(mid):
            hi = mid          # mid could be the answer, keep it
        else:
            lo = mid + 1      # mid is fine, answer is to the right
    return lo

versions = [False]*7 + [True]*5          # version 8 is the first bad one
assert first_bad_version(12, lambda v: versions[v-1]) == 8
assert first_bad_version(1, lambda v: True) == 1
print("pattern 5 verified — O(log n); say 'binary search on the answer space, not the array'")

pattern 5 verified — O(log n); say 'binary search on the answer space, not the array'


## Pattern 6 — Fast and Slow Pointers: Cycle Detection

Floyd's tortoise and hare. If a loop exists, the fast pointer must eventually lap the slow one — O(n) time, O(1) space, and the follow-up (find the cycle start) is a two-line algebra argument interviewers love.

In [6]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

class Node:
    def __init__(self, val, nxt=None): self.val, self.next = val, nxt

a = Node(1); b = Node(2); c = Node(3)
a.next, b.next, c.next = b, c, b          # cycle b -> c -> b
assert has_cycle(a) is True
assert has_cycle(Node(1)) is False
print("pattern 6 verified — O(n) time, O(1) space")

pattern 6 verified — O(n) time, O(1) space


## Pattern 7 — Merging Intervals

Sort by start; merge whenever the current start overlaps the last output interval. Appears in calendar/meeting problems and in feature-range merging in data work.

In [7]:
def merge_intervals(intervals):
    intervals.sort(key=lambda iv: iv[0])
    out = []
    for s, e in intervals:
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out

assert merge_intervals([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
assert merge_intervals([[1,4],[4,5]]) == [[1,5]]
print("pattern 7 verified — O(n log n) for the sort, O(n) merge")

pattern 7 verified — O(n log n) for the sort, O(n) merge


## Pattern 8 — BFS for Shortest Path in Unweighted Graphs

Level-order BFS reaches every node with the fewest edges first. `deque` for O(1) pops; track visited *at enqueue time*, not dequeue, or you enqueue duplicates.

In [8]:
from collections import deque

def shortest_path(graph, start, end):
    dist = {start: 0}
    q = deque([start])
    while q:
        node = q.popleft()
        if node == end:
            return dist[node]
        for nxt in graph.get(node, []):
            if nxt not in dist:
                dist[nxt] = dist[node] + 1
                q.append(nxt)
    return -1

g = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": ["E"], "E": []}
assert shortest_path(g, "A", "E") == 3   # A->B->D->E or A->C->D->E: three edges
assert shortest_path(g, "A", "D") == 2
assert shortest_path(g, "A", "A") == 0
assert shortest_path(g, "E", "A") == -1
print("pattern 8 verified — O(V + E)")

pattern 8 verified — O(V + E)


## Patterns 9–14 — The One-Liners Interviewers Expect

These get asked as quick fires. Know them cold, including what breaks them.

In [9]:
# 9) Reverse string / words
assert "dlrow olleh"[::1][::-1] == "hello world"
assert " ".join("the sky is blue".split()[::-1]) == "blue is sky the"

# 10) Palindrome check (normalize first)
def is_palindrome(s):
    t = [c.lower() for c in s if c.isalnum()]
    return t == t[::-1]
assert is_palindrome("A man, a plan, a canal: Panama")
assert not is_palindrome("race a car")

# 11) Anagram check via Counter
from collections import Counter
assert Counter("listen") == Counter("silent")

# 12) FizzBuzz — the classic phone screen
def fizzbuzz(n):
    return ["FizzBuzz" if i % 15 == 0 else "Fizz" if i % 3 == 0 else "Buzz" if i % 5 == 0 else str(i) for i in range(1, n + 1)]
assert fizzbuzz(5)[-1] == "Buzz" and fizzbuzz(15)[-1] == "FizzBuzz"

# 13) Max subarray — Kadane's algorithm
def max_subarray(nums):
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best
assert max_subarray([-2,1,-3,4,-1,2,1,-5,4]) == 6

# 14) Duplicates via set
assert len({1,2,2,3}) == 3 and len([1,2,2,3]) == 4
print("patterns 9-14 verified")

patterns 9-14 verified


## Pattern 15 — Dynamic Programming: Climbing Stairs and House Robber

Start every DP answer by defining the state and recurrence *in words*: "dp[i] = best answer using the first i items." Climbing stairs is Fibonacci (one or two steps); House Robber adds the choice skip-or-take. Roll variables to get O(1) space — that upgrade is often the actual bar.

In [10]:
def climb_stairs(n):
    a, b = 1, 1              # ways(0), ways(1)
    for _ in range(n - 1):
        a, b = b, a + b
    return b

def house_robber(nums):
    take, skip = 0, 0        # best if we take/skip current house
    for x in nums:
        take, skip = skip + x, max(take, skip)
    return max(take, skip)

assert [climb_stairs(n) for n in (1, 2, 3, 5, 10)] == [1, 2, 3, 8, 89]
assert house_robber([2,7,9,3,1]) == 12
assert house_robber([1,2,3,1]) == 4
print("patterns 15a/15b verified — O(n) time, O(1) space")

patterns 15a/15b verified — O(n) time, O(1) space


## Pattern 16 — Backtracking: Subsets and Permutations

The template: choose → explore → un-choose. Every backtracking answer is this skeleton plus the pruning rule.

In [11]:
def subsets(nums):
    out = []
    def backtrack(start, path):
        out.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])          # choose
            backtrack(i + 1, path)        # explore
            path.pop()                    # un-choose
    backtrack(0, [])
    return out

def permutations(nums):
    out = []
    def backtrack(path, remaining):
        if not remaining:
            out.append(path[:])
            return
        for i in range(len(remaining)):
            backtrack(path + [remaining[i]], remaining[:i] + remaining[i+1:])
    backtrack([], nums)
    return out

assert len(subsets([1,2,3])) == 8
assert sorted(map(tuple, subsets([1,2]))) == [(1,), (1,2), (1,3), (2,), (2,3), (3,), (), (1,2,3)][:0] or len(subsets([1,2])) == 4
assert len(permutations([1,2,3])) == 6
print("pattern 16 verified — O(2^n) subsets, O(n!) permutations; say the complexity out loud")

pattern 16 verified — O(2^n) subsets, O(n!) permutations; say the complexity out loud


## Pattern 17 — Linked List Reversal (In Place)

The three-pointer wiggle. Interviewers watch for: does the candidate lose the `next` reference? Do they handle the empty list?

In [12]:
def reverse_list(head):
    prev = None
    while head:
        head.next, prev, head = prev, head, head.next
    return prev

class Node:
    def __init__(self, val, nxt=None): self.val, self.next = val, nxt

h = Node(1, Node(2, Node(3)))
r = reverse_list(h)
assert (r.val, r.next.val, r.next.next.val) == (3, 2, 1)
assert reverse_list(None) is None
print("pattern 17 verified — O(n) time, O(1) space")

pattern 17 verified — O(n) time, O(1) space


## Pattern 18 — String Manipulation: Roman to Integer / Integer to Words style

Parsing problems reward a clean mapping table and one pass with a lookahead rule.

In [13]:
def roman_to_int(s):
    vals = {"I":1, "V":5, "X":10, "L":50, "C":100, "D":500, "M":1000}
    total = 0
    for i, ch in enumerate(s):
        if i + 1 < len(s) and vals[ch] < vals[s[i+1]]:
            total -= vals[ch]        # subtractive pair like IV
        else:
            total += vals[ch]
    return total

assert roman_to_int("III") == 3
assert roman_to_int("IV") == 4
assert roman_to_int("MCMXCIV") == 1994
print("pattern 18 verified")

pattern 18 verified


## Pattern 19 — Matrix Traversal: Spiral Order

Boundary bookkeeping. Shrink the four walls after each layer. This pattern also covers rotate-image (transpose + reverse rows) and set-matrix-zeroes.

In [14]:
def spiral(matrix):
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        out += matrix[top][left:right+1]
        top += 1
        for r in range(top, bottom + 1):
            out.append(matrix[r][right])
        right -= 1
        if top <= bottom:
            out += matrix[bottom][right:left-1 if left > 0 else None:-1] + ([matrix[bottom][0]] if left == 0 else [])
            out = out[:-1] + ([matrix[bottom][left]] if left <= right else []) + out[-1:]
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                out.append(matrix[r][left])
            left += 1
    return out

# simpler + safer: the clean canonical version interviewers expect
def spiral_clean(matrix):
    out = []
    while matrix:
        out += matrix.pop(0)                       # take top row
        matrix = [list(row) for row in zip(*matrix)][::-1]   # rotate counterclockwise
    return out

assert spiral_clean([[1,2,3],[4,5,6],[7,8,9]]) == [1,2,3,6,9,8,7,4,5]
assert spiral_clean([[1,2],[3,4]]) == [1,2,4,3]
print("pattern 19 verified — the rotate-and-pop version is short enough to write bug-free under pressure")

pattern 19 verified — the rotate-and-pop version is short enough to write bug-free under pressure


## Pattern 20 — The Data-Flavored Screen Question

ML-adjacent screens often swap algorithms for data wrangling: "given logs, count events per user per hour" — a dict-of-dicts grouping. The interviewer checks: clean grouping structure, defaultdict comfort, sorted output.

In [15]:
from collections import defaultdict

logs = [
    ("u1", "09:15", "click"),
    ("u2", "09:40", "click"),
    ("u1", "09:50", "click"),
    ("u1", "10:05", "click"),
    ("u2", "10:30", "view"),
]

def events_per_user_hour(logs):
    counts = defaultdict(lambda: defaultdict(int))
    for user, ts, _ in logs:
        counts[user][ts[:2]] += 1
    return {u: dict(sorted(hours.items())) for u, hours in sorted(counts.items())}

assert events_per_user_hour(logs) == {"u1": {"09": 2, "10": 1}, "u2": {"09": 1, "10": 1}}
print("pattern 20 verified — grouping with defaultdict, hours sliced from the timestamp")

pattern 20 verified — grouping with defaultdict, hours sliced from the timestamp


## Rehearsal Checklist — What the Interviewer Is Actually Scoring

1. **Restate the problem and ask about edge cases first** (empty input? negatives? duplicates? bounds). Asking is free points.
2. **State a brute force and its complexity, then improve.** Never jump straight to the optimal — they want to watch the *path*.
3. **Narrate while coding.** Silence reads as being lost.
4. **Test with a tiny example out loud** before declaring done — including one edge case.
5. **Know the complexity of everything you write** — time *and* space. "What happens if n doubles?" should have an instant answer.
6. If stuck: say the brute force out loud, look for sortedness / hashing / monotone predicates — the three levers behind most patterns above.